In [3]:
import numpy as np
import pandas as pd
from pathlib import Path

In [6]:
path = Path(
    "/Users/riku.personal/4th year school/"
    "Matrix and Tensor Techniques for DS/MandTtechForDS/"
    "LA county info/LAozone.txt"
)

columns = [
    "STATION", "MAXDAY", "AV8TOP", "MONITOR",
    "LAT", "LON", "X_COORD", "Y_COORD"
]

df = pd.read_csv(path, sep=r"\s+", skiprows=25, names=columns)
print(df.shape[0])

32


In [9]:
test = df.sample(n=6, random_state=42)
train = df.drop(test.index)

features = ["X_COORD", "Y_COORD"]

mean = train[features].mean()
std = train[features].std(ddof=0)

X_train = (train[features] - mean) / std
X_test = (test[features] - mean) / std

A_train = np.column_stack([np.ones(len(train)), X_train])
A_test = np.column_stack([np.ones(len(test)), X_test])
y_train = train["AV8TOP"].to_numpy()

beta = np.linalg.lstsq(A_train, y_train, rcond=None)[0]
predictions = A_test @ beta

results = test[["STATION", "AV8TOP"]].copy()
results.columns = ["STATION", "Actual"]
results["Predicted"] = predictions
results["Absolute_error"] = abs(results["Actual"] - predictions)

print(results.round(3).to_string(index=False))

rmse = np.sqrt(np.mean((results["Actual"] - predictions) ** 2))
baseline_rmse = np.sqrt(
    np.mean((results["Actual"] - y_train.mean()) ** 2)
)

print(f"\nModel RMSE:    {rmse:.3f}")
print(f"Baseline RMSE: {baseline_rmse:.3f}")

 STATION  Actual  Predicted  Absolute_error
    5212   8.524      7.254           1.270
    3186   4.601      5.339           0.738
    5175   8.149      7.387           0.762
    4137   8.242      9.535           1.293
      88   6.532      5.940           0.592
      89   7.540      5.927           1.613

Model RMSE:    1.108
Baseline RMSE: 1.434
